# SHACL Shape-Scoped Subgraph Extraction — User Guide

`StarShaclValidator.extract_subgraph()` is a fourth processing mode, alongside `validate()`, `apply_rules()`, and `evaluate()`: given a focus node that conforms to a shape, it extracts exactly the subgraph of *real, stored* triples that shape's constraints actually covered for that node - not the whole graph around it, and not a computed/virtual value invented along the way.

The motivating problem: suppose you record a signed hash of a journal entry (a "JE") so you can later verify it hasn't changed. If you later add an unrelated note to that same JE, a hash of "everything currently on that node" would change - even though nothing about the JE itself changed. `extract_subgraph()` lets you hash *only* the part a shape actually defines as "the JE", so unrelated additions don't affect it.

This guide builds up the feature one concept at a time, then ties it together with `starlayergraph.rdfc` (canonical hashing) in the final section, which is the actual end-to-end story this feature exists for.

## How to run this notebook

1. `pip install "git+https://github.com/hidden-graph/starlayer.git"` (or install the three packages editable from a local checkout — see the main user guide).
2. Run cells from top to bottom — later sections reuse the running example data from earlier ones.

In [1]:
from starlayergraph import StarLayerGraph, Namespace
from starshacl import StarShaclValidator, close_shape

EX = Namespace("http://example.org/")

## 1. Basic extraction

`extract_subgraph(data_graph=..., shacl_graph=..., shape=..., focus_node=...)` returns a `SubgraphExtractionResult` with two fields: `conforms` (did `focus_node` actually conform to `shape`?) and `data_graph` (the extracted subgraph, or `None` if `conforms` is `False`).

Here, `ex:je1` has an `ex:amount` and `ex:account` (both covered by `ex:JEShape`) and an `ex:note` (not mentioned anywhere in the shape). The note should not survive extraction.

**Example 1**

In [2]:
#the note is not part of the shape, so it must not appear in the extracted subgraph.
data = StarLayerGraph()
data.parse(data="""
    @prefix ex: <http://example.org/> .
    ex:je1 a ex:JournalEntry ; ex:amount 100 ; ex:account ex:acct1 ; ex:note "reviewed by Bob" .
""", format="turtle12")

shapes = StarLayerGraph()
shapes.parse(data="""
    @prefix ex: <http://example.org/> .
    @prefix sh: <http://www.w3.org/ns/shacl#> .
    ex:JEShape a sh:NodeShape ; sh:targetClass ex:JournalEntry ;
      sh:property [ sh:path ex:amount ; sh:minCount 1 ] ;
      sh:property [ sh:path ex:account ; sh:minCount 1 ] .
""", format="turtle12")

result = StarShaclValidator().extract_subgraph(
    data_graph=data, shacl_graph=shapes, shape=EX.JEShape, focus_node=EX.je1
)
print("conforms:", result.conforms)
print(result.data_graph.serialize(format="turtle12"))

conforms: True
@prefix ns1: <http://example.org/> .
@prefix xsd: <http://www.w3.org/2001/XMLSchema#> .

ns1:je1 ns1:account ns1:acct1 ;
    ns1:amount 100 .



## 2. The closed-shape acceptance test

How do you know an extraction is *exactly* right - not missing something the shape needed, and not carrying along anything extra? A useful check: take the extracted subgraph, and validate it against the *same* shape but closed, with **no `sh:ignoredProperties`** - not even ones the shape's normal, everyday production version might carry. If extraction is correct, this must still conform.

That "no ignored properties" part matters, not just as a technicality: real closed shapes very often *do* need `sh:ignoredProperties` - e.g. exempting `rdf:type`, since almost every real instance has one even though it's not usually declared via `sh:property`. If the acceptance test carried that same exemption over, a genuine extraction bug affecting exactly the ignored property could pass unnoticed. So rather than hand-writing a second, drift-prone copy of the shape, `starshacl.close_shape(shacl_graph, shape)` *derives* the strict version directly from the production one: it returns a copy with the shape (and everything it recursively references via `sh:node`/`sh:qualifiedValueShape`/`sh:and`/`sh:or`/`sh:xone`) closed, with any `sh:ignoredProperties` stripped - the original shape is never modified.

**Example 2**

In [3]:
#ex:JEShape's normal, everyday PRODUCTION form: closed, but explicitly
#tolerating rdf:type and free-form notes as known, legitimate extras.
prod_shapes = StarLayerGraph()
prod_shapes.parse(data="""
    @prefix ex: <http://example.org/> .
    @prefix sh: <http://www.w3.org/ns/shacl#> .
    @prefix rdf: <http://www.w3.org/1999/02/22-rdf-syntax-ns#> .
    ex:JEShape a sh:NodeShape ; sh:targetClass ex:JournalEntry ; sh:closed true ;
      sh:ignoredProperties ( rdf:type ex:note ) ;
      sh:property [ sh:path ex:amount ; sh:minCount 1 ] ;
      sh:property [ sh:path ex:account ; sh:minCount 1 ] .
""", format="turtle12")

result = StarShaclValidator().extract_subgraph(
    data_graph=data, shacl_graph=prod_shapes, shape=EX.JEShape, focus_node=EX.je1
)
print("conforms against the production shape:", result.conforms)

#derive the strict, no-exemptions verification shape FROM the production one -
#no separately hand-maintained shape to keep in sync.
strict_shapes = close_shape(prod_shapes, EX.JEShape)
print(strict_shapes.serialize(format="turtle12"))

check = StarShaclValidator().validate(data_graph=result.data_graph, shacl_graph=strict_shapes, meta_shacl=False)
print("extracted subgraph conforms to the derived strict shape:", check.conforms, "- expect True")
print("(no exemptions were needed - extraction already dropped both rdf:type and the note)")

conforms against the production shape: True
@prefix ex: <http://example.org/> .
@prefix sh: <http://www.w3.org/ns/shacl#> .
@prefix xsd: <http://www.w3.org/2001/XMLSchema#> .

ex:JEShape a sh:NodeShape ;
    sh:closed true ;
    sh:property [
        sh:minCount 1 ;
        sh:path ex:amount
    ], [
        sh:minCount 1 ;
        sh:path ex:account
    ] ;
    sh:targetClass ex:JournalEntry .

extracted subgraph conforms to the derived strict shape: True - expect True
(no exemptions were needed - extraction already dropped both rdf:type and the note)


## 3. Comparing extractions: isomorphism

Now that section 2 has verified extraction is exactly right, what happens when the underlying data actually changes over time? Extract "before" and "after", and compare them with `starlayergraph.compare.isomorphic()` (the same graph-isomorphism check used elsewhere in this project, correct for RDF 1.2 triple terms) - no hashing needed yet, just a direct graph comparison.

Adding an unrelated note shouldn't change what gets extracted, so the two extractions should still be isomorphic. Actually changing the amount should change what gets extracted, so they should no longer be isomorphic.

**Example 3**

In [4]:
from starlayergraph.compare import isomorphic

shapes = StarLayerGraph()
shapes.parse(data="""
    @prefix ex: <http://example.org/> .
    @prefix sh: <http://www.w3.org/ns/shacl#> .
    ex:JEShape a sh:NodeShape ; sh:targetClass ex:JournalEntry ;
      sh:property [ sh:path ex:amount ; sh:minCount 1 ] ;
      sh:property [ sh:path ex:account ; sh:minCount 1 ] .
""", format="turtle12")

#the "before" extraction
data_before = StarLayerGraph()
data_before.parse(data="@prefix ex: <http://example.org/> . ex:je1 a ex:JournalEntry ; ex:amount 100 ; ex:account ex:acct1 .", format="turtle12")
before = StarShaclValidator().extract_subgraph(data_graph=data_before, shacl_graph=shapes, shape=EX.JEShape, focus_node=EX.je1)

#a note gets added - unrelated to the shape
data_note = StarLayerGraph()
data_note.parse(data='''
    @prefix ex: <http://example.org/> .
    ex:je1 a ex:JournalEntry ; ex:amount 100 ; ex:account ex:acct1 ; ex:note "reviewed" .
''', format="turtle12")
after_note = StarShaclValidator().extract_subgraph(data_graph=data_note, shacl_graph=shapes, shape=EX.JEShape, focus_node=EX.je1)
print("note added, extractions still isomorphic:", isomorphic(before.data_graph, after_note.data_graph), "- expect True")

#the amount actually changes - a real edit
data_changed = StarLayerGraph()
data_changed.parse(data="@prefix ex: <http://example.org/> . ex:je1 a ex:JournalEntry ; ex:amount 999 ; ex:account ex:acct1 .", format="turtle12")
after_changed = StarShaclValidator().extract_subgraph(data_graph=data_changed, shacl_graph=shapes, shape=EX.JEShape, focus_node=EX.je1)
print("amount changed, extractions isomorphic:  ", isomorphic(before.data_graph, after_changed.data_graph), "- expect False")

note added, extractions still isomorphic: True - expect True
amount changed, extractions isomorphic:   False - expect False


## 4. Checking a pending addition before it lands

A common real workflow: proposed changes sit in a separate "pending" graph before being merged into the live "data" graph - e.g. an ingestion pipeline staging incoming triples for review. Before committing the merge, you can check whether it's actually *safe* with respect to a shape you care about: extract now (before), simulate the merge, extract again (after), and compare with `isomorphic()` - all without touching the real data graph yet.

`starlayergraph.StarLayerDataset` holds multiple named graphs - `data` (the current, committed state) and `pending` (the staged addition) - inside one dataset, exactly the "one graph holds the addition, one holds the current graph" shape this workflow needs.

An addition that's genuinely irrelevant to the shape (another note) leaves the extraction isomorphic - safe to merge. An addition that lands on a path the shape already covers - here, a *second* `ex:amount` value for the same JE - changes what gets extracted (both amounts are now present), so the check correctly reports it's *not* isomorphic, flagging the merge as unsafe before it ever touches the real data graph.

**Example 4**

In [5]:
from starlayergraph import StarLayerDataset
from starlayergraph.compare import isomorphic

shapes = StarLayerGraph()
shapes.parse(data="""
    @prefix ex: <http://example.org/> .
    @prefix sh: <http://www.w3.org/ns/shacl#> .
    ex:JEShape a sh:NodeShape ; sh:targetClass ex:JournalEntry ;
      sh:property [ sh:path ex:amount ; sh:minCount 1 ] ;
      sh:property [ sh:path ex:account ; sh:minCount 1 ] .
""", format="turtle12")


def check_pending_is_safe_to_merge(pending_ttl, label):
    ds = StarLayerDataset()
    data_g = ds.graph(EX.data)
    data_g.parse(data="@prefix ex: <http://example.org/> . ex:je1 a ex:JournalEntry ; ex:amount 100 ; ex:account ex:acct1 .", format="turtle12")
    pending_g = ds.graph(EX.pending)
    pending_g.parse(data=pending_ttl, format="turtle12")

    before = StarShaclValidator().extract_subgraph(data_graph=data_g, shacl_graph=shapes, shape=EX.JEShape, focus_node=EX.je1)

    #simulate the merge - a throwaway graph, the real data_g is untouched.
    simulated = StarLayerGraph()
    for t in data_g:
        simulated.add(t)
    for t in pending_g:
        simulated.add(t)
    after = StarShaclValidator().extract_subgraph(data_graph=simulated, shacl_graph=shapes, shape=EX.JEShape, focus_node=EX.je1)

    safe = isomorphic(before.data_graph, after.data_graph)
    print(f"{label}: safe to merge = {safe}")
    return safe


check_pending_is_safe_to_merge(
    '@prefix ex: <http://example.org/> . ex:je1 ex:note "reviewed" .',
    "pending = an unrelated note",
)
check_pending_is_safe_to_merge(
    "@prefix ex: <http://example.org/> . ex:je1 ex:amount 999 .",
    "pending = a SECOND, conflicting amount",
)

pending = an unrelated note: safe to merge = True
pending = a SECOND, conflicting amount: safe to merge = False


False

## 5. Multi-hop paths

A `sh:path` can be more than a single predicate - a sequence like `(ex:employee ex:name)` follows two hops. Extraction keeps *every* intervening triple along the way, not just the final value - here, both `ex:alice ex:employee ex:bob` and `ex:bob ex:name "Bob"` are needed to justify the path, so both are included. `ex:bob`'s own unrelated `ex:secret` is not.

**Example 5**

In [6]:
#a two-hop sequence path - both hops are kept, ex:bob's unrelated
#ex:secret is not.
data = StarLayerGraph()
data.parse(data="""
    @prefix ex: <http://example.org/> .
    ex:alice ex:employee ex:bob ; ex:unrelated "noise" .
    ex:bob ex:name "Bob" ; ex:secret "shh" .
""", format="turtle12")

shapes = StarLayerGraph()
shapes.parse(data="""
    @prefix ex: <http://example.org/> .
    @prefix sh: <http://www.w3.org/ns/shacl#> .
    ex:S a sh:NodeShape ; sh:targetNode ex:alice ;
      sh:property [ sh:path (ex:employee ex:name) ; sh:minCount 1 ] .
""", format="turtle12")

result = StarShaclValidator().extract_subgraph(data_graph=data, shacl_graph=shapes, shape=EX.S, focus_node=EX.alice)
print(result.data_graph.serialize(format="turtle12"))

@prefix ns1: <http://example.org/> .

ns1:alice ns1:employee ns1:bob .

ns1:bob ns1:name "Bob" .



## 6. Nested shapes

A property shape can require its values to conform to a *second* shape via `sh:node`. Extraction recurses into that nested shape too - whatever real triples *it* required are included, following the same rules recursively.

**Example 6**

In [7]:
#ex:PersonShape's own requirements on ex:bob (his ex:name) are pulled in
#recursively; ex:bob's unrelated ex:secret is not.
data = StarLayerGraph()
data.parse(data="""
    @prefix ex: <http://example.org/> .
    ex:alice ex:employee ex:bob .
    ex:bob ex:name "Bob" ; ex:secret "shh" .
""", format="turtle12")

shapes = StarLayerGraph()
shapes.parse(data="""
    @prefix ex: <http://example.org/> .
    @prefix sh: <http://www.w3.org/ns/shacl#> .
    ex:PersonShape a sh:NodeShape ; sh:property [ sh:path ex:name ; sh:minCount 1 ] .
    ex:S a sh:NodeShape ; sh:targetNode ex:alice ;
      sh:property [ sh:path ex:employee ; sh:node ex:PersonShape ] .
""", format="turtle12")

result = StarShaclValidator().extract_subgraph(data_graph=data, shacl_graph=shapes, shape=EX.S, focus_node=EX.alice)
print(result.data_graph.serialize(format="turtle12"))

@prefix ns1: <http://example.org/> .

ns1:alice ns1:employee ns1:bob .

ns1:bob ns1:name "Bob" .



## 7. Logical constraints: `sh:or` and `sh:xone`

`sh:or` only requires *at least one* disjunct to pass, so more than one may legitimately conform at the same time - extraction includes every disjunct that passes, not just the first in list order (list order is an authoring artifact, not a semantic tie-break).

`sh:xone` is different: it requires *exactly one* disjunct to pass, so at most one can ever legitimately conform - extraction includes that one. This isn't an arbitrary determinism choice the way picking the first `sh:or` branch used to be; it's the only value that can ever validly exist. See Example 7.2 for what happens if data later changes so a second `sh:xone` disjunct also passes.

**Example 7.1**

In [8]:
#alice satisfies both ex:HasEmail and ex:HasPhone - sh:or includes both
#branches' triples, not just the first.
data = StarLayerGraph()
data.parse(data="""
    @prefix ex: <http://example.org/> .
    ex:alice ex:email "a@example.org" ; ex:phone "555" .
""", format="turtle12")

shapes = StarLayerGraph()
shapes.parse(data="""
    @prefix ex: <http://example.org/> .
    @prefix sh: <http://www.w3.org/ns/shacl#> .
    ex:HasEmail a sh:NodeShape ; sh:property [ sh:path ex:email ; sh:minCount 1 ] .
    ex:HasPhone a sh:NodeShape ; sh:property [ sh:path ex:phone ; sh:minCount 1 ] .
    ex:S a sh:NodeShape ; sh:targetNode ex:alice ; sh:or ( ex:HasEmail ex:HasPhone ) .
""", format="turtle12")

result = StarShaclValidator().extract_subgraph(data_graph=data, shacl_graph=shapes, shape=EX.S, focus_node=EX.alice)
print(result.data_graph.serialize(format="turtle12"))


@prefix ns1: <http://example.org/> .

ns1:alice ns1:email "a@example.org" ;
    ns1:phone "555" .



With `sh:xone` in place of `sh:or` on the same two shapes, only one disjunct may pass. While that holds, extraction behaves like `sh:or` with a single passer:

**Example 7.2**

In [9]:
#with sh:xone, alice having ONLY an email conforms (exactly one disjunct passes)
#and extraction includes that one branch.
shapes_xone = StarLayerGraph()
shapes_xone.parse(data="""
    @prefix ex: <http://example.org/> .
    @prefix sh: <http://www.w3.org/ns/shacl#> .
    ex:HasEmail a sh:NodeShape ; sh:property [ sh:path ex:email ; sh:minCount 1 ] .
    ex:HasPhone a sh:NodeShape ; sh:property [ sh:path ex:phone ; sh:minCount 1 ] .
    ex:S a sh:NodeShape ; sh:targetNode ex:alice ; sh:xone ( ex:HasEmail ex:HasPhone ) .
""", format="turtle12")

email_only = StarLayerGraph()
email_only.parse(data='@prefix ex: <http://example.org/> . ex:alice ex:email "a@example.org" .', format="turtle12")

before = StarShaclValidator().extract_subgraph(data_graph=email_only, shacl_graph=shapes_xone, shape=EX.S, focus_node=EX.alice)
print("conforms:", before.conforms)
print(before.data_graph.serialize(format="turtle12"))

#now alice also gets a phone number - a SECOND xone disjunct passes, which
#means sh:xone itself ("exactly one") is no longer satisfied. This is not a
#different subgraph - it's outright non-conformance.
email_and_phone = StarLayerGraph()
email_and_phone.parse(data="""
    @prefix ex: <http://example.org/> .
    ex:alice ex:email "a@example.org" ; ex:phone "555" .
""", format="turtle12")

after = StarShaclValidator().extract_subgraph(data_graph=email_and_phone, shacl_graph=shapes_xone, shape=EX.S, focus_node=EX.alice)
print("conforms:", after.conforms, " data_graph:", after.data_graph)


conforms: True
@prefix ns1: <http://example.org/> .

ns1:alice ns1:email "a@example.org" .

conforms: False  data_graph: None


## 8. Qualified value shapes and determinism

`sh:qualifiedValueShape` + `sh:qualifiedMinCount` says "at least N values must conform to this shape." If *more* than N actually do, which one(s) "count"? There's no principled single answer - all of them equally contribute - so extraction includes **every** candidate that qualifies, not an arbitrary minimum-sized subset. This trades minimality for determinism: the result never depends on backend iteration order, which matters because this feature feeds a hash (see section 11).

Below, `ex:alice` has three employees; `ex:bob` and `ex:carol` are in Sales (both qualify), `ex:dave` is in Engineering (doesn't). Both qualifying employees are included - `ex:dave` is excluded *entirely*, since nothing else on the property shape needs him.

**Example 8.1**

In [10]:
#both Sales employees (bob, carol) qualify and are both included;
#dave (Eng) doesn't qualify and is excluded entirely.
data = StarLayerGraph()
data.parse(data="""
    @prefix ex: <http://example.org/> .
    ex:alice ex:employee ex:bob, ex:carol, ex:dave .
    ex:bob ex:dept ex:Sales . ex:carol ex:dept ex:Sales . ex:dave ex:dept ex:Eng .
""", format="turtle12")

shapes = StarLayerGraph()
shapes.parse(data="""
    @prefix ex: <http://example.org/> .
    @prefix sh: <http://www.w3.org/ns/shacl#> .
    ex:S a sh:NodeShape ; sh:targetNode ex:alice ;
      sh:property [ sh:path ex:employee ;
                    sh:qualifiedValueShape [ sh:property [ sh:path ex:dept ; sh:hasValue ex:Sales ] ] ;
                    sh:qualifiedMinCount 1 ] .
""", format="turtle12")

result = StarShaclValidator().extract_subgraph(data_graph=data, shacl_graph=shapes, shape=EX.S, focus_node=EX.alice)
print(result.data_graph.serialize(format="turtle12"))

@prefix ns1: <http://example.org/> .

ns1:alice ns1:employee ns1:carol, ns1:bob .

ns1:bob ns1:dept ns1:Sales .

ns1:carol ns1:dept ns1:Sales .



If something *else* on the same property shape needs the full, unfiltered set of values - e.g. a plain `sh:minCount` on the raw path, counting *all* employees regardless of department - narrowing doesn't happen, and `ex:dave` is correctly included again.

**Example 8.2**

In [11]:
#sh:minCount 3 on the raw path needs ALL employees, so dave is included
#even though he doesn't satisfy the qualifying shape.
shapes2 = StarLayerGraph()
shapes2.parse(data="""
    @prefix ex: <http://example.org/> .
    @prefix sh: <http://www.w3.org/ns/shacl#> .
    ex:S a sh:NodeShape ; sh:targetNode ex:alice ;
      sh:property [ sh:path ex:employee ; sh:minCount 3 ;
                    sh:qualifiedValueShape [ sh:property [ sh:path ex:dept ; sh:hasValue ex:Sales ] ] ;
                    sh:qualifiedMinCount 1 ] .
""", format="turtle12")

result = StarShaclValidator().extract_subgraph(data_graph=data, shacl_graph=shapes2, shape=EX.S, focus_node=EX.alice)
print(result.data_graph.serialize(format="turtle12"))

@prefix ns1: <http://example.org/> .

ns1:alice ns1:employee ns1:bob, ns1:dave, ns1:carol .

ns1:bob ns1:dept ns1:Sales .

ns1:carol ns1:dept ns1:Sales .



## 9. Cycles and unbounded paths

`sh:zeroOrMorePath`/`sh:oneOrMorePath` can reach an unbounded, even cyclic, set of nodes. Extraction handles real cycles correctly (it terminates and captures every edge among the reachable set, including a self-loop) rather than looping forever.

**Example 9**

In [12]:
#alice -> bob -> alice (a cycle) and carol -> carol (a self-loop) -
#extraction terminates and captures every friend-edge among the reachable
#set; the unrelated ex:dave is excluded.
data = StarLayerGraph()
data.parse(data="""
    @prefix ex: <http://example.org/> .
    ex:alice ex:friend ex:bob .
    ex:bob ex:friend ex:alice, ex:carol .
    ex:carol ex:friend ex:carol .
    ex:dave a ex:Unrelated .
""", format="turtle12")

shapes = StarLayerGraph()
shapes.parse(data="""
    @prefix ex: <http://example.org/> .
    @prefix sh: <http://www.w3.org/ns/shacl#> .
    ex:S a sh:NodeShape ; sh:targetNode ex:alice ;
      sh:property [ sh:path [ sh:zeroOrMorePath ex:friend ] ; sh:minCount 1 ] .
""", format="turtle12")

result = StarShaclValidator().extract_subgraph(data_graph=data, shacl_graph=shapes, shape=EX.S, focus_node=EX.alice)
print(result.data_graph.serialize(format="turtle12"))

@prefix ns1: <http://example.org/> .

ns1:alice ns1:friend ns1:bob .

ns1:bob ns1:friend ns1:carol, ns1:alice .

ns1:carol ns1:friend ns1:carol .



## 10. Non-conformance

`extract_subgraph()` assumes `focus_node` already conforms to `shape` - it's meant to extract, not to diagnose. If it doesn't conform, `conforms` is `False` and `data_graph` is `None`, with no report explaining why (the caller is expected to have checked conformance already, e.g. via `validate()`).

**Example 10**

In [13]:
#ex:je2 is missing both ex:amount and ex:account, so it doesn't conform.
data = StarLayerGraph()
data.parse(data="@prefix ex: <http://example.org/> . ex:je2 a ex:JournalEntry .", format="turtle12")

shapes = StarLayerGraph()
shapes.parse(data="""
    @prefix ex: <http://example.org/> .
    @prefix sh: <http://www.w3.org/ns/shacl#> .
    ex:JEShape a sh:NodeShape ; sh:targetClass ex:JournalEntry ;
      sh:property [ sh:path ex:amount ; sh:minCount 1 ] ;
      sh:property [ sh:path ex:account ; sh:minCount 1 ] .
""", format="turtle12")

result = StarShaclValidator().extract_subgraph(data_graph=data, shacl_graph=shapes, shape=EX.JEShape, focus_node=EX.je2)
print("conforms:", result.conforms, "- expect False")
print("data_graph:", result.data_graph, "- expect None")

conforms: False - expect False
data_graph: None - expect None


## 11. Putting it together: hashing a subgraph

A bare hash isn't quite enough to reverify later: the hash alone doesn't say *which* shape produced it, so a verifier re-extracting with the wrong shape (or a shape that's since drifted) has no way to know that's what happened - it would just look like the data changed. So what actually gets recorded is a small **commitment** - a real, named RDF graph, not just a Python value - holding the focus node, the shape used, when it was committed, and a hash that binds the focus node and the shape to the extracted subgraph, not just the subgraph's content on its own.

There are two distinct graphs worth seeing separately: the **envelope** - the extracted subgraph plus one triple binding it to the shape - is what actually gets canonicalized and hashed; the **commitment record** is the durable artifact that gets kept, naming that hash alongside the focus node, the shape, and a timestamp. The commitment is given its own real identity - `urn:starlayer:ledger#commitment-<hash>`, content-addressed by the very hash it carries, not a blank node.

The link between the two only needs to be stored in one direction: the commitment points *to* its focus node via `stledger:focusNode` (the standard "record points to its subject" shape, like a Verifiable Credential's `credentialSubject` - it also keeps the commitment self-describing if it's ever exported or signed on its own, away from the live data). "Which commitment does this node have?" doesn't need a second, separately-asserted reverse triple - that would just be the same fact stored twice, with nothing to keep the two in sync if one is ever updated without the other. It's answered by querying the existing triple instead: `commitment.subjects(stledger:focusNode, je1)`.

`commit_subgraph()` below builds the envelope, hashes it with `starlayergraph.rdfc.rdfc10_hash()`, and returns the commitment record. `verify_commitment()` reads the focus node and shape back out of that record, re-extracts under them, and checks the same envelope still hashes the same way. (Eventually the commitment graph itself should be signed, so it can't be quietly edited after the fact - that's out of scope for now.)

The walkthrough below is a full lifecycle for one JE, tying this together with section 4's staging pattern:

1. Stage the JE's *initial* creation in `pending` and check it would conform before it ever touches `data`.
2. Merge it into `data` and commit - this produces the commitment record everything downstream checks against.
3. Stage an unrelated note, confirm it's safe (still isomorphic), merge it, and confirm the *original* commitment still verifies.
4. Stage a conflicting second `amount`, find it's *not* safe, and correctly refuse to merge it - the commitment is never even at risk.
5. Apply that same conflicting edit directly to `data`, bypassing staging entirely - now verification only catches it *after the fact*.
6. Leave the data alone and instead add a new required property to `ex:JEShape` itself - verification fails again, this time because the shape no longer conforms, not because the subgraph's hash changed. Same outward signal (verification failed), different root cause - see the discussion above.

**Example 11**

In [14]:
from datetime import datetime, timezone

from starlayergraph import StarLayerDataset, Literal, RDF
from starlayergraph.compare import isomorphic
from starlayergraph.rdfc import rdfc10_hash

STLEDGER = Namespace("urn:starlayer:ledger#")

def build_envelope(data_graph, shacl_graph, shape, focus_node):
    """The subgraph a shape covers, plus one triple binding it to *this*
    shape - this is what actually gets canonicalized and hashed."""
    result = StarShaclValidator().extract_subgraph(
        data_graph=data_graph, shacl_graph=shacl_graph, shape=shape, focus_node=focus_node
    )
    if not result.conforms:
        return None
    envelope = StarLayerGraph()
    for t in result.data_graph:
        envelope.add(t)
    envelope.add((focus_node, STLEDGER.committedUnderShape, shape))
    return envelope

def commit_subgraph(data_graph, shacl_graph, shape, focus_node):
    """Hash the envelope and return the durable commitment record - a
    named node (content-addressed by its own hash, not a blank node),
    linked to FROM the validated node via stledger:hasCommitment."""
    envelope = build_envelope(data_graph, shacl_graph, shape, focus_node)
    if envelope is None:
        return None
    hash_value = rdfc10_hash(envelope)

    commitment_node = STLEDGER[f"commitment-{hash_value}"]
    commitment = StarLayerGraph()
    commitment.add((commitment_node, RDF.type, STLEDGER.Commitment))
    commitment.add((commitment_node, STLEDGER.focusNode, focus_node))
    commitment.add((commitment_node, STLEDGER.shape, shape))
    commitment.add((commitment_node, STLEDGER.hash, Literal(hash_value)))
    commitment.add((commitment_node, STLEDGER.committedAt, Literal(datetime.now(timezone.utc))))
    return commitment

def verify_commitment(commitment, data_graph, shacl_graph):
    """Read the focus node/shape back out of the record, re-build the
    envelope, and recompute the same hash."""
    node = next(commitment.subjects(RDF.type, STLEDGER.Commitment))
    focus_node = commitment.value(node, STLEDGER.focusNode)
    shape = commitment.value(node, STLEDGER.shape)
    stored_hash = str(commitment.value(node, STLEDGER.hash))

    envelope = build_envelope(data_graph, shacl_graph, shape, focus_node)
    if envelope is None:
        return False
    return rdfc10_hash(envelope) == stored_hash

shapes = StarLayerGraph()
shapes.parse(data="""
    @prefix ex: <http://example.org/> .
    @prefix sh: <http://www.w3.org/ns/shacl#> .
    ex:JEShape a sh:NodeShape ; sh:targetClass ex:JournalEntry ;
      sh:property [ sh:path ex:amount ; sh:minCount 1 ] ;
      sh:property [ sh:path ex:account ; sh:minCount 1 ] .
""", format="turtle12")

#1. stage the JE's initial creation - nothing in `data` yet, so there's no
#"before" to compare against; the test is simply "would the merged result conform".
ds = StarLayerDataset()
data_g = ds.graph(EX.data)
pending_g = ds.graph(EX.pending)
pending_g.parse(data="""
    @prefix ex: <http://example.org/> .
    ex:je1 a ex:JournalEntry ; ex:amount 100 ; ex:account ex:acct1 .
""", format="turtle12")

simulated = StarLayerGraph()
for t in data_g: simulated.add(t)
for t in pending_g: simulated.add(t)
precheck = StarShaclValidator().extract_subgraph(data_graph=simulated, shacl_graph=shapes, shape=EX.JEShape, focus_node=EX.je1)
print("1. staged creation would conform:", precheck.conforms)

#2. merge, then commit - show BOTH the envelope that gets hashed and the
#commitment record that gets kept, plus the link between them.
for t in list(pending_g):
    data_g.add(t)
pending_g.remove((None, None, None))

print("2. what was committed (the envelope):")
print(build_envelope(data_g, shapes, EX.JEShape, EX.je1).serialize(format="turtle12"))

commitment = commit_subgraph(data_g, shapes, EX.JEShape, EX.je1)
print("2. the commitment record:")
print(commitment.serialize(format="turtle12"))
print("2. je1's commitment, found by querying the record's own focusNode:", list(commitment.subjects(STLEDGER.focusNode, EX.je1)))

#3. stage an unrelated note - safe, merge it, original commitment still holds
pending_g.parse(data='@prefix ex: <http://example.org/> . ex:je1 ex:note "reviewed by Bob" .', format="turtle12")
before = StarShaclValidator().extract_subgraph(data_graph=data_g, shacl_graph=shapes, shape=EX.JEShape, focus_node=EX.je1)
simulated = StarLayerGraph()
for t in data_g: simulated.add(t)
for t in pending_g: simulated.add(t)
after = StarShaclValidator().extract_subgraph(data_graph=simulated, shacl_graph=shapes, shape=EX.JEShape, focus_node=EX.je1)
safe = isomorphic(before.data_graph, after.data_graph)
print("3. unrelated note safe to merge:", safe)
if safe:
    for t in list(pending_g): data_g.add(t)
    pending_g.remove((None, None, None))
print("3. commitment still verifies after merging it:", verify_commitment(commitment, data_g, shapes))

#keep a clean, known-good snapshot to reuse independently in steps 5 and 6
data_good = StarLayerGraph()
for t in data_g: data_good.add(t)

#4. stage a CONFLICTING second amount - not safe, so it's never merged
pending_g.parse(data="@prefix ex: <http://example.org/> . ex:je1 ex:amount 999 .", format="turtle12")
before = StarShaclValidator().extract_subgraph(data_graph=data_g, shacl_graph=shapes, shape=EX.JEShape, focus_node=EX.je1)
simulated = StarLayerGraph()
for t in data_g: simulated.add(t)
for t in pending_g: simulated.add(t)
after = StarShaclValidator().extract_subgraph(data_graph=simulated, shacl_graph=shapes, shape=EX.JEShape, focus_node=EX.je1)
safe = isomorphic(before.data_graph, after.data_graph)
print("4. conflicting amount safe to merge:", safe, "- refusing to merge")
pending_g.remove((None, None, None))  # discard - data_g is never touched
print("4. data untouched, commitment still verifies:", verify_commitment(commitment, data_g, shapes))

#5. the SAME conflicting edit, but applied directly to data - no staging,
#no chance to catch it beforehand. Verification is the only thing left to catch it.
data_bypassed = StarLayerGraph()
for t in data_good: data_bypassed.add(t)
data_bypassed.parse(data="@prefix ex: <http://example.org/> . ex:je1 ex:amount 999 .", format="turtle12")
print("5. verify after an unstaged direct edit:", verify_commitment(commitment, data_bypassed, shapes))

#6. data is untouched (data_good) - instead, ex:JEShape itself gains a new
#required property je1 doesn't have. Same "verification failed" signal,
#but this time via non-conformance, not a hash mismatch.
shapes_v2 = StarLayerGraph()
shapes_v2.parse(data="""
    @prefix ex: <http://example.org/> .
    @prefix sh: <http://www.w3.org/ns/shacl#> .
    ex:JEShape a sh:NodeShape ; sh:targetClass ex:JournalEntry ;
      sh:property [ sh:path ex:amount ; sh:minCount 1 ] ;
      sh:property [ sh:path ex:account ; sh:minCount 1 ] ;
      sh:property [ sh:path ex:currency ; sh:minCount 1 ] .
""", format="turtle12")
print("6. verify under a since-modified shape:", verify_commitment(commitment, data_good, shapes_v2))


1. staged creation would conform: True
2. what was committed (the envelope):
@prefix ns1: <http://example.org/> .
@prefix ns2: <urn:starlayer:ledger#> .
@prefix xsd: <http://www.w3.org/2001/XMLSchema#> .

ns1:je1 ns1:account ns1:acct1 ;
    ns1:amount 100 ;
    ns2:committedUnderShape ns1:JEShape .

2. the commitment record:
@prefix ns1: <urn:starlayer:ledger#> .
@prefix ns2: <http://example.org/> .
@prefix xsd: <http://www.w3.org/2001/XMLSchema#> .

ns1:commitment-36d19fd32f9d73c31460b87873e7c785f2df865cf48d3d0971593626dd6bc4e3 a ns1:Commitment ;
    ns1:committedAt "2026-09-13T10:30:10.840665+00:00"^^xsd:dateTime ;
    ns1:focusNode ns2:je1 ;
    ns1:hash "36d19fd32f9d73c31460b87873e7c785f2df865cf48d3d0971593626dd6bc4e3" ;
    ns1:shape ns2:JEShape .

2. je1's commitment, found by querying the record's own focusNode: [rdflib.term.URIRef('urn:starlayer:ledger#commitment-36d19fd32f9d73c31460b87873e7c785f2df865cf48d3d0971593626dd6bc4e3')]
3. unrelated note safe to merge: True
3. commitm

## Further work

- **`sh:not` contributes no additional triples.** There's no well-defined "witness" of a shape *not* matching, unlike the other logical constraints - this is a deliberate simplification, not yet tested against real fixtures using `sh:not`.
- **Complex paths combined with `sh:qualifiedValueShape`** (a sequence/alternative/`zeroOrMore` path, rather than a simple or inverse one) fall back to including the full, unfiltered value set rather than precisely pruning down to just the qualifying candidates - precise pruning there is a harder graph-reachability problem not yet attempted.
- **Shape "blast radius" awareness.** A shape's reach can be larger and less obvious than it looks - multi-hop paths and nested shapes mean a seemingly small, unrelated edit several hops away can change a hash. There's currently no tooling to show a shape author exactly how far their shape reaches before they rely on it for hashing; `extract_subgraph()` itself, run during shape design, is one way to see this today, but a dedicated warning/analysis tool for "dangerous" shape constructs doesn't exist yet.
- **Protecting *existing* commitments against unrelated writes isn't feasible in general.** Section 11's staging pattern pre-checks a write safely because it already knows which commitment it's protecting - extract before/after for that one `(shape, focus_node)` and compare. An arbitrary write elsewhere in the graph carries no such label, so the only exhaustive way to guarantee nothing broke would be re-running every existing commitment's before/after check on every single write anywhere in the graph - O(all live commitments) per write, which doesn't scale and isn't done. A real fix would need a reverse dependency index - per commitment, the set of nodes actually *visited* during its own extraction traversal (not just the triples that ended up included, since a new value hanging off an already-visited node, like a new qualifying employee, can change the result without touching any previously-included triple) - so an arbitrary write only needs checking against commitments whose dependency set it actually intersects. Not built, and likely never will be - noted here as an accepted, understood limitation rather than an open TODO.
- **Shape and extractor versioning.** A hash is really a function of (data, shape, focus node, extractor implementation) - if the shape is later edited, or a future bug fix changes what `extract_subgraph()` considers relevant, previously-recorded hashes won't reproduce even though nothing about "the JE itself" changed. Section 11's commitment binds the shape's *identity* (its IRI), which catches a verifier using the wrong shape entirely - but it doesn't hash the shape's own *definition*, so a shape silently redefined under the same IRI (that happens to still extract the same triples) wouldn't be caught. Fully closing that gap would mean canonicalizing and hashing the shape's own triples into the commitment too - not done here.
- **Signing.** The commitment's hash isn't signed yet, so the commitment record itself could be edited after the fact with nothing to catch it. The intent is to sign the hash (e.g. with the recording party's key) so the commitment - not just the underlying data - is tamper-evident; deliberately out of scope for this guide.